# KaKaRoTiOS — Dragon Ball Z: Kakarot

Pipeline isolado na branch `KaKaRoTiOS`, a partir de `SpkIOSzero`. O Colab baixa **seis volumes RAR**: três do jogo base e três do update. Cada conjunto é testado e extraído separadamente pelo RARLAB UnRAR; o resultado precisa ser exatamente um NSP ou XCI íntegro por conjunto. Base, update, RomFS e AOC são armazenados separadamente no Drive; o workspace de build também fica separado.

**Os volumes RAR só são apagados depois que os dois pacotes reconstruídos foram enviados e verificados no Drive.** A última célula é um despacho explícito e não executa por padrão.

In [ ]:
# Cole os seis links diretos, na ordem dos volumes 1, 2 e 3.
DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"

BASE_PART_URLS = [
    "",  # base volume 1
    "",  # base volume 2
    "",  # base volume 3
]
UPDATE_PART_URLS = [
    "",  # update volume 1
    "",  # update volume 2
    "",  # update volume 3
]

# Opcional: nomes originais exatos dos volumes se o servidor não os enviar
# no Content-Disposition/URL. Deixe os três vazios para detecção automática.
# RAR moderno: game.part1.rar, game.part2.rar, game.part3.rar
# RAR antigo: game.rar, game.r00, game.r01
BASE_PART_NAMES = ["", "", ""]
UPDATE_PART_NAMES = ["", "", ""]

# Não ativa o GitHub Actions nesta etapa. O despacho é uma célula separada.
DISPATCH_TO_GITHUB = False

In [ ]:
from pathlib import Path
import argparse, json, os, subprocess, sys
from google.colab import auth

auth.authenticate_user()
ROOT = Path("/content/kakarot-ios-repo")
if not ROOT.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "KaKaRoTiOS",
        "https://github.com/joao9vegeto-stack/KintTany.git", str(ROOT)
    ], check=True)
else:
    branch = subprocess.check_output(
        ["git", "-C", str(ROOT), "branch", "--show-current"], text=True
    ).strip()
    if branch != "KaKaRoTiOS":
        raise RuntimeError(f"Workspace existente está na branch {branch!r}; não vou trocar de branch.")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "KaKaRoTiOS"], check=True)
    subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "origin/KaKaRoTiOS"], check=True)

assert subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip() == "KaKaRoTiOS"
sys.path.insert(0, str(ROOT))
PY_FILES = [
    "colab/source_kakarot.py", "colab/prepare_kakarot.py", "colab/run_kakarot.py",
    "scripts/prepare_kakarot_actions.py", "scripts/build_kakarot_shared.py",
]
subprocess.run([sys.executable, "-m", "py_compile", *[str(ROOT / item) for item in PY_FILES]], check=True)
print("KaKaRoTiOS Python syntax: OK")
print("Repository:", ROOT)
print("Branch:", "KaKaRoTiOS")

In [ ]:
from colab.run_kakarot import prepare, recover

if all(x.strip() for x in BASE_PART_URLS + UPDATE_PART_URLS):
    handoff = prepare(argparse.Namespace(
        drive_folder_id=DRIVE_FOLDER_ID,
        base_part_urls=[x.strip() for x in BASE_PART_URLS],
        update_part_urls=[x.strip() for x in UPDATE_PART_URLS],
        base_part_names=BASE_PART_NAMES if any(x.strip() for x in BASE_PART_NAMES) else None,
        update_part_names=UPDATE_PART_NAMES if any(x.strip() for x in UPDATE_PART_NAMES) else None,
    ))
else:
    print("Os seis links ainda não estão preenchidos. Tentando recuperar uma preparação anterior do Drive...")
    handoff = recover(argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID))
    if handoff is None:
        raise ValueError("Preencha os três BASE_PART_URLS e os três UPDATE_PART_URLS, na ordem volume 1, 2, 3.")

print("\nPreparação recuperada/concluída:", handoff["request_id"])
print("Drive:", handoff["drive_output_folder_url"])

In [ ]:
# Auditoria antes de qualquer despacho para GitHub Actions.
prepared = handoff["prepared"]
print(json.dumps({
    "project": handoff["project"],
    "branch": handoff["branch"],
    "title_id": handoff["title_id"],
    "package_version": handoff["package_version"],
    "base_drive": handoff["game_data"]["base_package"],
    "update_drive": handoff["game_data"]["update_package"],
    "rar_parts_deleted": handoff["source"]["rar_parts_deleted_after_upload"],
    "profile_type": prepared["profile_type"],
    "profile_source": prepared["profile_source"],
    "main_build_id": prepared["main_build_id"],
    "main_sha256": prepared["main_sha256"],
    "modules": prepared["modules"],
    "graphics_api": prepared["graphics_api"],
    "unreal_shader_sources": prepared["unreal_shader_sources"],
    "pak_audit": prepared["pak_audit"],
    "shader_chain": prepared["shader_chain"],
    "romfs_size": prepared["romfs_size"],
    "romfs_sha256": prepared["romfs_sha256"],
    "aoc_count": prepared["aoc_count"],
    "aoc_images": prepared["aoc_images"],
    "drive_workspace": handoff["work_archive"],
}, indent=2))

In [ ]:
# DESPACHO EXPLÍCITO — só execute depois de revisar o handoff acima.
if DISPATCH_TO_GITHUB:
    from colab.run_kakarot import dispatch
    dispatch(argparse.Namespace(handoff=Path("/content/kakarot-ios/handoff.json")))
else:
    print("Actions NÃO disparada. Para enviar, altere DISPATCH_TO_GITHUB = True e execute esta célula.")